## Middleware in Langchain

Middleware provides a way to more tightly control what happens inside the agent 

In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver #--> moved from langchain to langgraph
from langchain_core.messages import HumanMessage, SystemMessage

agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("messages", 10),
            keep=("messages", 4)
)
    ]
)

## Human In the Loop Middleware

In [2]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.agents.middleware import HumanInTheLoopMiddleware


def read_email_tool(email_id: str) -> str:
    '''Mock function to read an email by its ID'''
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str):
    '''Mock function to send email to recepient'''
    return f"Email sent to {recipient} with subject: {subject} and {body}"
    

In [5]:
agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={
            "send_email_tool" : {
                "allowed_decisions": ["approved", "edit", "reject"]
            },
            "read_email_tool": False
        }
    )]
)